# Polars vs. Pandas: In-Memory Architecture, Lazy Execution, and Benchmarks

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

While Pandas has served as the bedrock of Python data analysis for over a decade, modern datasets expose fundamental architectural limitations in its design. Polars was engineered from scratch in Rust to overcome these limitations.

### Core Architectural Contrasts

| Dimension | Pandas | Polars |
| :--- | :--- | :--- |
| **Primary Language** | Python + C / Cython | Rust |
| **Memory Model** | NumPy BlockManager / Object Pointers | Apache Arrow Columnar Format |
| **Concurrency** | Single-threaded (constrained by Python GIL) | Multi-threaded work-stealing (Rayon) |
| **Execution Engine** | Eager evaluation only | Eager (`DataFrame`) and Lazy (`LazyFrame`) |
| **Optimization** | None (sequential imperative execution) | Predicate & projection pushdown, query graph pruning |
| **SIMD Support** | Limited / compiler-dependent | Native vectorized SIMD instruction kernels |
| **Memory Overhead** | High ($2\times$ to $5\times$ dataset size in RAM) | Low (contiguous memory buffers, zero-copy slicing) |

### Learning Objectives
1. Contrast the memory layouts of Pandas and Polars.
2. Construct and optimize execution graphs using Polars `LazyFrame`.
3. Inspect query optimization via `explain()` (predicate and projection pushdown).
4. Conduct empirical runtime benchmarks comparing ingestion, filtering, and aggregation throughput.
5. Apply migration recipes translating common Pandas patterns into Polars expressions.


In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

print(f"Polars Version: {pl.__version__}")
print(f"Pandas Version: {pd.__version__}")

## 1. Lazy Execution and the `LazyFrame`

In eager execution (`pl.read_csv` or `pd.read_csv`), every line of code immediately processes the full dataset into memory. In contrast, **Lazy Execution** (`pl.scan_csv`) constructs a directed acyclic graph (DAG) of logical operations without loading data:

1. **Predicate Pushdown:** Polars moves `.filter()` calls down to the file reading layer, only loading rows that satisfy the condition.
2. **Projection Pushdown:** Polars identifies which columns are used in downstream computations and skips reading unused columns from disk.
3. **Common Subexpression Elimination:** Shared expressions are evaluated once rather than recomputed.


In [ ]:
# Create a synthetic benchmark CSV dataset
BENCH_CSV = Path("synthetic_transactions.csv")
N_RECORDS = 150_000

np.random.seed(42)
customer_ids = np.random.randint(1000, 5000, size=N_RECORDS)
product_categories = np.random.choice(
    ["Electronics", "Apparel", "Home", "Books", "Beauty"], size=N_RECORDS
)
amounts = np.round(np.random.exponential(scale=50.0, size=N_RECORDS) + 5.0, 2)
discounts = np.round(np.random.uniform(0.0, 0.3, size=N_RECORDS), 2)
ratings = np.random.choice([1, 2, 3, 4, 5], size=N_RECORDS)

bench_df = pl.DataFrame(
    {
        "customer_id": customer_ids,
        "category": product_categories,
        "amount": amounts,
        "discount": discounts,
        "rating": ratings,
    }
)

bench_df.write_csv(BENCH_CSV)
file_size_mb = BENCH_CSV.stat().st_size / (1024 * 1024)
print(f"Synthesized {N_RECORDS:,} records in '{BENCH_CSV}' ({file_size_mb:.2f} MB)")

### 1.1 Inspecting the Query Plan with `.explain()`

We build a lazy query and examine how the Polars optimizer transforms the execution plan.


In [ ]:
# Create LazyFrame using scan_csv
lazy_plan = (
    pl.scan_csv(BENCH_CSV)
    .filter(pl.col("amount") > 75.0)
    .filter(pl.col("category") == "Electronics")
    .select(["customer_id", "category", "amount", "discount"])
    .with_columns((pl.col("amount") * (1 - pl.col("discount"))).alias("net_revenue"))
    .group_by("customer_id")
    .agg(pl.col("net_revenue").sum().alias("total_customer_spend"))
)

print("Optimized Logical Plan:")
print(lazy_plan.explain())

### 1.2 Materializing Results with `.collect()`

The query graph is executed only when `.collect()` is invoked:


In [ ]:
t0 = time.perf_counter()
materialized_result = lazy_plan.collect()
lazy_elapsed = time.perf_counter() - t0

print(f"Lazy query executed in {lazy_elapsed * 1000:.2f} ms")
print(f"Result shape: {materialized_result.shape}")
print(materialized_result.head(4))

## 2. Empirical Benchmarking: Polars vs. Pandas

We benchmark three fundamental operations across both libraries:
1. **CSV Ingestion:** Reading raw data from disk.
2. **Filtering & Arithmetic:** Subsetting rows and computing derived features.
3. **Group-By Aggregation:** Grouping by categorical keys and computing statistics.


In [ ]:
# Benchmark 1: CSV Ingestion
# Pandas
t0 = time.perf_counter()
pd_df = pd.read_csv(BENCH_CSV)
pd_ingest_time = time.perf_counter() - t0

# Polars (Eager)
t0 = time.perf_counter()
pl_df = pl.read_csv(BENCH_CSV)
pl_ingest_time = time.perf_counter() - t0

# Benchmark 2: Filtering and Column Computation
# Pandas
t0 = time.perf_counter()
pd_filtered = pd_df[(pd_df["amount"] > 50.0) & (pd_df["rating"] >= 4)].copy()
pd_filtered["net_amount"] = pd_filtered["amount"] * (1 - pd_filtered["discount"])
pd_filter_time = time.perf_counter() - t0

# Polars
t0 = time.perf_counter()
pl_filtered = pl_df.filter((pl.col("amount") > 50.0) & (pl.col("rating") >= 4)).with_columns(
    (pl.col("amount") * (1 - pl.col("discount"))).alias("net_amount")
)
pl_filter_time = time.perf_counter() - t0

# Benchmark 3: Group-by Aggregation
# Pandas
t0 = time.perf_counter()
pd_grouped = pd_df.groupby("category").agg(
    mean_amount=("amount", "mean"), total_amount=("amount", "sum"), count=("amount", "count")
)
pd_group_time = time.perf_counter() - t0

# Polars
t0 = time.perf_counter()
pl_grouped = pl_df.group_by("category").agg(
    [
        pl.col("amount").mean().alias("mean_amount"),
        pl.col("amount").sum().alias("total_amount"),
        pl.len().alias("count"),
    ]
)
pl_group_time = time.perf_counter() - t0

print(f"Ingestion Speedup:      {pd_ingest_time / pl_ingest_time:.2f}x faster in Polars")
print(f"Filter/Compute Speedup: {pd_filter_time / pl_filter_time:.2f}x faster in Polars")
print(f"Group-by Speedup:       {pd_group_time / pl_group_time:.2f}x faster in Polars")

## 3. Visualizing Benchmark Comparisons

We plot the execution latency across all three benchmark tasks.


In [ ]:
tasks = ["CSV Ingestion", "Filter & Compute", "Group-by Aggregation"]
pd_times = [pd_ingest_time * 1000, pd_filter_time * 1000, pd_group_time * 1000]
pl_times = [pl_ingest_time * 1000, pl_filter_time * 1000, pl_group_time * 1000]

x = np.arange(len(tasks))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
rects1 = ax.bar(
    x - width / 2,
    pd_times,
    width,
    label="Pandas (ms)",
    color="#d62728",
    alpha=0.85,
    edgecolor="black",
)
rects2 = ax.bar(
    x + width / 2,
    pl_times,
    width,
    label="Polars (ms)",
    color="#1f77b4",
    alpha=0.85,
    edgecolor="black",
)

ax.set_ylabel("Execution Time (milliseconds) - Lower is Better")
ax.set_title("Performance Comparison: Pandas vs. Polars (150,000 Records)")
ax.set_xticks(x)
ax.set_xticklabels(tasks)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# Clean up synthetic benchmark data
if BENCH_CSV.exists():
    BENCH_CSV.unlink()
print(f"Removed temporary file {BENCH_CSV}.")

## 4. Pandas to Polars Migration Playbook

When migrating existing machine learning and data pipelines from Pandas to Polars, reference the following canonical translation table:

| Operation | Pandas Syntax | Polars Expression Syntax |
| :--- | :--- | :--- |
| **Select Columns** | `df[['a', 'b']]` | `df.select(['a', 'b'])` |
| **Filter Rows** | `df[(df['a'] > 0) & (df['b'] == 1)]` | `df.filter((pl.col('a') > 0) & (pl.col('b') == 1))` |
| **Add/Modify Column** | `df['c'] = df['a'] * 2` | `df.with_columns((pl.col('a') * 2).alias('c'))` |
| **Rename Column** | `df.rename(columns={'old': 'new'})` | `df.rename({'old': 'new'})` |
| **Sort Values** | `df.sort_values('a', ascending=False)` | `df.sort('a', descending=True)` |
| **Group By** | `df.groupby('k').agg({'v': 'mean'})` | `df.group_by('k').agg(pl.col('v').mean())` |
| **Table Join** | `df.merge(df2, on='k', how='left')` | `df.join(df2, on='k', how='left')` |
| **Drop Duplicates** | `df.drop_duplicates(subset=['a'])` | `df.unique(subset=['a'])` |
| **Handle Missing** | `df.dropna()` / `df.fillna(0)` | `df.drop_nulls()` / `df.fill_null(0)` |


## 5. Summary and Key Takeaways

1. **Memory Efficiency:** Apache Arrow columnar formatting eliminates the pointer-chasing overhead of traditional Python objects and enables SIMD vectorization.
2. **Lazy Optimization:** `pl.scan_csv()` and `.explain()` unlock query plan optimizations such as predicate pushdown and projection pushdown, minimizing disk I/O and RAM usage.
3. **Concurrency Without GIL:** Multi-threaded execution scales across available CPU hardware without Python multiprocessing complexity.
4. **Clean Expression API:** Polars decouples expressions from execution contexts, producing readable, composable, and mathematically sound data pipelines.
